# Settings

In [ ]:
! pip install 'smolagents[openai]'
! pip install 'smolagents[toolkit]'
! pip install nba_api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.5 MB/s eta 0:00:00
  Attempting uninstall: lxml
    Found existing installation: lxml 5.4.0
    Uninstalling lxml-5.4.0:
      Successfully uninstalled lxml-5.4.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.0/287.0 kB 6.3 MB/s eta 0:00:00


In [ ]:
# Basic
from google.colab import userdata
import os
import json
from typing import List, Dict, Any
import pandas as pd

# smolagents
from smolagents import CodeAgent, OpenAIServerModel, DuckDuckGoSearchTool, FinalAnswerTool, tool, Tool

# NBA API
from nba_api.live.nba.endpoints import scoreboard
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.static import players

# Gradio
import gradio as gr

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

# Tools

In [ ]:
@tool
def get_todays_records_summary() -> List[Dict[str, Any]]:
    """
    Retrieves and summarizes today's NBA game scores and results.
    Use this tool ONLY when the user explicitly asks about today's NBA games, scores, results, or summaries.
    This function requires no parameters.

    Returns:
        A list of dictionaries, where each dictionary represents one game.
        If no games are found, returns an empty list.
        If the data fetch fails, returns a list containing a single error dictionary.

    Each game dictionary has the following structure:
    {
        "gameId": "0022500028",
        "gameStatus": "Final",
        "homeTeam": {"name": "Magic", "tricode": "ORL", "score": 123},
        "awayTeam": {"name": "Celtics", "tricode": "BOS", "score": 110},
        "winnerTricode": "ORL",
        "gameLeaders": {
            "home": {"name": "Franz Wagner", "points": 27, "rebounds": 6, "assists": 6},
            "away": {"name": "Jaylen Brown", "points": 32, "rebounds": 9, "assists": 3}
        }
    }
    """
    try:
        # Attempt to fetch the main scoreboard data
        today_games_data = scoreboard.ScoreBoard().get_dict()['scoreboard']['games']
    except Exception as e:
        print(f"Error fetching scoreboard data: {e}")
        # Return a list containing an error dict if the API call itself fails
        return [{"error": f"Failed to retrieve NBA data: {e}"}]

    if not today_games_data:
        # If no games are scheduled, return an empty list. This is not an error.
        return []

    summaries: List[Dict[str, Any]] = []
    for game in today_games_data:
        try:
            # --- 1. Safely extract nested data ---
            # Use .get() to avoid KeyErrors if a key is missing
            home_team = game.get('homeTeam', {})
            away_team = game.get('awayTeam', {})
            game_leaders = game.get('gameLeaders', {})
            home_leader = game_leaders.get('homeLeaders', {})
            away_leader = game_leaders.get('awayLeaders', {})

            # --- 2. Calculate score and determine winner ---
            home_score = home_team.get('score', 0)
            away_score = away_team.get('score', 0)
            # Get game status (e.g., "Final", "In Progress", "Scheduled")
            game_status = game.get('gameStatusText', 'Scheduled')

            winner_tricode = 'N/A'
            # Only compare scores if they are valid numbers
            if isinstance(home_score, (int, float)) and isinstance(away_score, (int, float)):
                if home_score > away_score:
                    winner_tricode = home_team.get('teamTricode', 'N/A')
                elif away_score > home_score:
                    winner_tricode = away_team.get('teamTricode', 'N/A')

            # --- 3. Build the "clean summary dictionary" for this game ---
            game_summary = {
                "gameId": game.get('gameId', 'N/A'),
                "gameStatus": game_status, # For user context (e.g., "is the game over?")

                # Requirement 1: Home and Away teams
                "homeTeam": {
                    "name": home_team.get('teamName', 'N/A'),
                    "city": home_team.get('teamCity', 'N/A'),
                    "tricode": home_team.get('teamTricode', 'N/A'),
                    "score": home_score  # Requirement 2: Score
                },
                "awayTeam": {
                    "name": away_team.get('teamName', 'N/A'),
                    "city": away_team.get('teamCity', 'N/A'),
                    "tricode": away_team.get('teamTricode', 'N/A'),
                    "score": away_score  # Requirement 2: Score
                },

                # Requirement 2: Game result (Winner)
                "winnerTricode": winner_tricode,
                # Requirement 3: Leaders and their stats (Points, etc.)
                "gameLeaders": {
                    "home": {
                        "name": home_leader.get('name', 'N/A').encode('latin1').decode('utf-8'),
                        "points": home_leader.get('points', 'N/A'),
                        "rebounds": home_leader.get('rebounds', 'N/A'), # Requirement 4: Extra info
                        "assists": home_leader.get('assists', 'N/A')  # Requirement 4: Extra info
                    },
                    "away": {
                        "name": away_leader.get('name', 'N/A').encode('latin1').decode('utf-8'),
                        "points": away_leader.get('points', 'N/A'),
                        "rebounds": away_leader.get('rebounds', 'N/A'),
                        "assists": away_leader.get('assists', 'N/A')
                    }
                }
            }
            summaries.append(game_summary)

        except Exception as e:
            # Handle errors for a single game (e.g., unexpected data format)
            game_id = game.get('gameId', 'Unknown ID')
            print(f"Error processing game {game_id}: {e}")
            summaries.append({"error": f"Error processing game {game_id}", "details": str(e)})

    return summaries

In [ ]:
@tool
def format_summaries_to_text(game_summaries: List[Dict[str, Any]]) -> str:
    """
    Formats a list of game summary dictionaries into a single, user-friendly text string.
    Use this tool when the user asks for a general summary of all games,
    and you ALREADY have the game data (List[Dict]) from 'get_todays_records_summary'.

    Args:
        game_summaries: The list of game dictionaries returned by
                         'get_todays_records_summary'.

    Returns:
        A formatted string summarizing all games.
    """
    if not game_summaries:
        return "No NBA games were found for today."

    # Check if the list contains an error message instead of game data
    if len(game_summaries) == 1 and "error" in game_summaries[0]:
        return f"An error occurred: {game_summaries[0]['error']}"

    output_lines = ["🏀 Today's NBA Game Results 🏀\n"]

    for game in game_summaries:
        # Handle individual game processing errors, if any
        if "error" in game:
            output_lines.append(f"--- \n[Error processing one game: {game.get('details', 'Unknown')}]")
            continue

        try:
            home = game.get('homeTeam', {})
            away = game.get('awayTeam', {})
            leaders = game.get('gameLeaders', {})
            home_leader = leaders.get('home', {})
            away_leader = leaders.get('away', {})

            # --- 1. Game Status and Score ---
            status = game.get('gameStatus', 'Scheduled')
            line = f"[{status}] {home.get('name')} ({home.get('score')}) vs {away.get('name')} ({away.get('score')})"

            # --- 2. Winner ---
            winner_tricode = game.get('winnerTricode', 'N/A')
            if status == 'Final':
                if winner_tricode == home.get('tricode'):
                    line += f" (Winner: {home.get('name')})"
                elif winner_tricode == away.get('tricode'):
                    line += f" (Winner: {away.get('name')})"

            output_lines.append(line)

            # --- 3. Leaders Info (Only if game is Final/In Progress) ---
            if status != 'Scheduled':
                output_lines.append(
                    f"  -> Home Leader: {home_leader.get('name', 'N/A')} "
                    f"({home_leader.get('points', 0)} pts, {home_leader.get('rebounds', 0)} reb, {home_leader.get('assists', 0)} ast)"
                )
                output_lines.append(
                    f"  -> Away Leader: {away_leader.get('name', 'N/A')} "
                    f"({away_leader.get('points', 0)} pts, {away_leader.get('rebounds', 0)} reb, {away_leader.get('assists', 0)} ast)"
                )

            output_lines.append("---") # Separator

        except Exception as e:
            # Catch errors during formatting
            output_lines.append(f"--- \n[Error formatting game {game.get('gameId', 'N/A')}: {e}]")


    return "\n".join(output_lines)

In [ ]:
@tool
def find_player_id(player_name: str) -> Dict[str, Any]:
    """
    Searches for an NBA player by name and returns a structured result.
    Use this to find a player's ID before calling other player-specific tools
    like 'get_player_career_stats'.

    Args:
        player_name: The full or partial name of the NBA player (e.g., "LeBron", "Curry").

    Returns:
        A dictionary describing the search result, which will have one of
        the following structures:

        1.  **Exact Match Found:**
            {
                "status": "FOUND",
                "player": {"id": 2544, "full_name": "LeBron James"}
                (Note: 'id' is returned as an 'int' for the next tool)
            }

        2.  **Multiple Matches Found (for disambiguation):**
            {
                "status": "MULTIPLE_FOUND",
                "matches": [
                    {"id": 2544, "full_name": "LeBron James"},
                    {"id": 1630547, "full_name": "LeBron James Jr."}
                ]
            }

        3.  **No Match Found:**
            {
                "status": "NOT_FOUND",
                "message": "No players found matching the name: 'PlayerName'"
            }

        4.  **Error:**
            {
                "status": "ERROR",
                "message": "An error occurred..."
            }
    """
    try:
        # Use the nba_api function to find players (returns a list of dicts)
        player_list = players.find_players_by_full_name(player_name)

        if not player_list:
            # Case 3: No Match Found
            return {
                "status": "NOT_FOUND",
                "message": f"No players found matching the name: '{player_name}'."
            }

        elif len(player_list) == 1:
            # Case 1: Exact Match Found
            player = player_list[0]
            return {
                "status": "FOUND",
                # Return the 'id' as an integer, because 'get_player_career_stats'
                # expects an 'int'. This saves the Agent a step.
                "player": {
                    "id": int(player['id']),
                    "full_name": player['full_name']
                }
            }

        else:
            # Case 2: Multiple Matches Found
            # Prepare a clean list for the Agent to show the user
            matches = [
                {
                    "id": int(p['id']),
                    "full_name": p['full_name']
                }
                for p in player_list
            ]

            # Optional: Limit the number of matches to avoid overwhelming the user/context
            if len(matches) > 10:
                matches = matches[:10]

            return {
                "status": "MULTIPLE_FOUND",
                "matches": matches
            }

    except Exception as e:
        # Case 4: Error
        print(f"Error in find_player_id for '{player_name}': {e}")
        return {
            "status": "ERROR",
            "message": f"An error occurred during the player search: {str(e)}"
        }

In [ ]:
def _process_stats_dataframe(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Helper function to process a stats DataFrame into a list of dicts."""
    # 1. Define key columns to keep
    columns_to_keep = [
        'SEASON_ID', 'TEAM_ABBREVIATION', 'PLAYER_AGE',
        'GP', 'GS', 'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
        'FG_PCT', 'FG3_PCT', 'FT_PCT'
    ]

    # 2. Filter for available columns (Career Totals df might not have all keys)
    available_columns = [col for col in columns_to_keep if col in df.columns]
    filtered_df = df[available_columns].copy() # Use .copy() to avoid SettingWithCopyWarning

    # 3. Rename columns to be agent-friendly
    column_rename_map = {
        'SEASON_ID': 'season',
        'TEAM_ABBREVIATION': 'team',
        'PLAYER_AGE': 'age',
        'GP': 'gp', 'GS': 'gs', 'MIN': 'min', 'PTS': 'pts', 'REB': 'reb',
        'AST': 'ast', 'STL': 'stl', 'BLK': 'blk', 'TOV': 'tov',
        'FG_PCT': 'fg_pct', 'FG3_PCT': 'fg3_pct', 'FT_PCT': 'ft_pct'
    }

    # 4. Apply rename mapping
    rename_mapping_applied = {k: v for k, v in column_rename_map.items() if k in filtered_df.columns}
    final_df = filtered_df.rename(columns=rename_mapping_applied)

    # 5. Convert to List[Dict]
    return final_df.to_dict('records')

@tool
def get_player_career_stats(player_id: int) -> Dict[str, Any]:
    """
    Retrieves season-by-season AND career total stats for an NBA player.
    Use this to get the raw data for analysis (e.g., "find his best season").

    Args:
        player_id: The unique NBA-assigned ID for the player.

    Returns:
        A dictionary with a 'status' key and data.

        1.  **Success:**
            {
                "status": "FOUND",
                "seasons": [
                    {"season": "2003-04", "team": "CLE", "pts": 20.9, ...},
                    ...
                ],
                "total": {
                    "gp": 1492, "pts": 25.7, "reb": 7.3, ...
                    (Note: This is ONE dict, not a list)
                }
            }

        2.  **Failure/Error:**
            {
                "status": "ERROR",
                "message": "Error message..."
            }
    """
    try:
        # Fetch all DataFrames provided by the endpoint
        all_data_frames = playercareerstats.PlayerCareerStats(player_id=player_id).get_data_frames()

        # [0] is Season-by-Season stats
        season_stats_df = all_data_frames[0]
        # [1] is Career Totals (as a single-row DataFrame)
        total_stats_df = all_data_frames[1]

        if season_stats_df.empty or total_stats_df.empty:
            return {
                "status": "NOT_FOUND",
                "message": f"No career stats found for player ID {player_id}."
            }

        # Process both DataFrames using the helper function
        seasons_list = _process_stats_dataframe(season_stats_df)

        # total_list will be a list with ONE element, so we extract it
        total_dict = _process_stats_dataframe(total_stats_df)[0]

        # Return the complete, structured data
        return {
            "status": "FOUND",
            "seasons": seasons_list,
            "total": total_dict
        }

    except Exception as e:
        print(f"Error processing stats for player ID {player_id}: {e}")
        return {
            "status": "ERROR",
            "message": f"Error retrieving stats for player ID {player_id}: {str(e)}"
        }

In [ ]:
@tool
def format_career_stats_to_text(player_name: str, career_data: Dict[str, Any]) -> str:
    """
    Formats the structured career stats (from get_player_career_stats)
    into a user-friendly, readable text string.

    Use this tool ONLY when the user wants to SEE the stats,
    after you have retrieved them with 'get_player_career_stats'.

    Args:
        player_name: The player's full name (e.g., "LeBron James").
        career_data: The dictionary output from 'get_player_career_stats'
                     (assumed to contain *total* stats, not per-game).

    Returns:
        A formatted string summarizing the player's career.
    """
    # --- 1. Handle Error or Not Found cases ---
    status = career_data.get("status")
    if status != "FOUND":
        return career_data.get("message", "Could not retrieve stats.")

    try:
        total = career_data.get("total", {})
        seasons = career_data.get("seasons", [])

        output_lines = [f"📊 **Career Stats Summary for {player_name}** 📊\n"]

        # --- 2. Format Career Totals (as Integers) ---

        output_lines.append("--- **Career Totals (All Regular Seasons)** ---")
        if total:

            total_line_1 = (
                f"  - **GP:** {int(total.get('gp', 0))} | "
                f"**MIN:** {int(total.get('min', 0))} | "
                f"**PTS:** {int(total.get('pts', 0))} | "
                f"**REB:** {int(total.get('reb', 0))} | "
                f"**AST:** {int(total.get('ast', 0))}"
            )

            total_line_2 = (
                f"  - **STL:** {int(total.get('stl', 0))} | "
                f"**BLK:** {int(total.get('blk', 0))} | "
                f"**TOV:** {int(total.get('tov', 0))} | "
                f"**FG%:** {total.get('fg_pct', 0) * 100:.1f}% | "
                f"**3P%:** {total.get('fg3_pct', 0) * 100:.1f}%"
            )
            output_lines.append(total_line_1)
            output_lines.append(total_line_2)
        else:
            output_lines.append("  (No total stats available)")

        # --- 3. Format Season-by-Season (as Integers) ---

        output_lines.append("\n--- **Season-by-Season (Totals)** ---")
        if not seasons:
            output_lines.append("  (No individual season stats available)")

        # Create a header (Expanded, for Totals)
        output_lines.append(
            "`Season  | Team | GP |  MIN |   PTS |   REB |   AST |   STL |   BLK |   TOV |   FG% |   3P%`"
        )

        for season in seasons:
            # Format key season stats (Integers and Floats)
            season_line = (
                f"`{season.get('season', 'N/A'):<8} | "
                f"{season.get('team', 'N/A'):<4} | "
                f"{int(season.get('gp', 0)):>3} | "
                f"{int(season.get('min', 0)):>4} | "
                f"{int(season.get('pts', 0)):>5} | "
                f"{int(season.get('reb', 0)):>5} | "
                f"{int(season.get('ast', 0)):>5} | "
                f"{int(season.get('stl', 0)):>5} | "
                f"{int(season.get('blk', 0)):>5} | "
                f"{int(season.get('tov', 0)):>5} | "
                f"{season.get('fg_pct', 0) * 100:>5.1f}% | "
                f"{season.get('fg3_pct', 0) * 100:>5.1f}%`"
            )
            output_lines.append(season_line)

        return "\n".join(output_lines)

    except Exception as e:
        print(f"Error formatting stats for {player_name}: {e}")
        return f"An error occurred while formatting stats for {player_name}."

In [ ]:
custom_tools = [get_todays_records_summary, format_summaries_to_text, find_player_id, get_player_career_stats, format_career_stats_to_text]

# Models and Agents

In [ ]:
model = OpenAIServerModel(
    model_id="gemini-2.5-flash",
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=GEMINI_API_KEY,
)

agent = CodeAgent(tools=[DuckDuckGoSearchTool(), FinalAnswerTool()] + custom_tools, model=model)

# Inference

In [ ]:
result = agent.run("What were today's NBA game results?")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What were today's NBA game results?                                                                             │
│                                                                                                                 │
╰─ OpenAIServerModel - gemini-2.5-flash ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  game_summaries = get_todays_records_summary()                                                                    
  print(game_summaries)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[{'gameId': '0022500028', 'gameStatus': 'Final', 'homeTeam': {'name': 'Magic', 'city': 'Orlando', 'tricode': 'ORL',
'score': 123}, 'awayTeam': {'name': 'Celtics', 'city': 'Boston', 'tricode': 'BOS', 'score': 110}, 'winnerTricode': 
'ORL', 'gameLeaders': {'home': {'name': 'Franz Wagner', 'points': 27, 'rebounds': 6, 'assists': 6}, 'away': 
{'name': 'Jaylen Brown', 'points': 32, 'rebounds': 9, 'assists': 3}}}, {'gameId': '0022500029', 'gameStatus': 
'Final', 'homeTeam': {'name': 'Wizards', 'city': 'Washington', 'tricode': 'WAS', 'score': 114}, 'awayTeam': 
{'name': 'Cavaliers', 'city': 'Cleveland', 'tricode': 'CLE', 'score': 148}, 'winnerTricode': 'CLE', 'gameLeaders': 
{'home': {'name': 'CJ McCollum', 'points': 25, 'rebounds': 2, 'assists': 3}, 'away': {'name': 'Donovan Mitchell', 
'points': 24, 'rebounds': 5, 'assists': 5}}}, {'gameId': '0022500030', 'gameStatus': 'Final', 'homeTeam': {'name': 
'Hawks', 'city': 'Atlanta', 'tricode': 'ATL', 'score': 97}, 'awayTeam': {'name': 'Raptors', 'city': 'Toronto', 
'tricode': 'TOR', 'score': 109}, 'winnerTricode': 'TOR', 'gameLeaders': {'home': {'name': 'Jalen Johnson', 
'points': 21, 'rebounds': 7, 'assists': 4}, 'away': {'name': 'Brandon Ingram', 'points': 20, 'rebounds': 6, 
'assists': 4}}}, {'gameId': '0022500031', 'gameStatus': 'Final', 'homeTeam': {'name': 'Nets', 'city': 'Brooklyn', 
'tricode': 'BKN', 'score': 107}, 'awayTeam': {'name': 'Pistons', 'city': 'Detroit', 'tricode': 'DET', 'score': 
125}, 'winnerTricode': 'DET', 'gameLeaders': {'home': {'name': 'Michael Porter Jr.', 'points': 28, 'rebounds': 5, 
'assists': 2}, 'away': {'name': 'Cade Cunningham', 'points': 34, 'rebounds': 1, 'assists': 10}}}, {'gameId': 
'0022500032', 'gameStatus': 'Final', 'homeTeam': {'name': 'Spurs', 'city': 'San Antonio', 'tricode': 'SAS', 
'score': 121}, 'awayTeam': {'name': 'Rockets', 'city': 'Houston', 'tricode': 'HOU', 'score': 110}, 'winnerTricode':
'SAS', 'gameLeaders': {'home': {'name': 'Victor Wembanyama', 'points': 22, 'rebounds': 8, 'assists': 4}, 'away': 
{'name': 'Alperen Sengun', 'points': 25, 'rebounds': 9, 'assists': 8}}}, {'gameId': '0022500033', 'gameStatus': 
'Final', 'homeTeam': {'name': 'Heat', 'city': 'Miami', 'tricode': 'MIA', 'score': 126}, 'awayTeam': {'name': 
'Hornets', 'city': 'Charlotte', 'tricode': 'CHA', 'score': 108}, 'winnerTricode': 'MIA', 'gameLeaders': {'home': 
{'name': 'Jaime Jaquez Jr.', 'points': 18, 'rebounds': 8, 'assists': 9}, 'away': {'name': 'Kon Knueppel', 'points':
30, 'rebounds': 8, 'assists': 3}}}, {'gameId': '0022500034', 'gameStatus': 'Final', 'homeTeam': {'name': 
'Grizzlies', 'city': 'Memphis', 'tricode': 'MEM', 'score': 118}, 'awayTeam': {'name': 'Mavericks', 'city': 
'Dallas', 'tricode': 'DAL', 'score': 104}, 'winnerTricode': 'MEM', 'gameLeaders': {'home': {'name': 'Ja Morant', 
'points': 21, 'rebounds': 5, 'assists': 13}, 'away': {'name': 'Naji Marshall', 'points': 16, 'rebounds': 7, 
'assists': 4}}}, {'gameId': '0022500035', 'gameStatus': 'Final', 'homeTeam': {'name': 'Bucks', 'city': 'Milwaukee',
'tricode': 'MIL', 'score': 126}, 'awayTeam': {'name': 'Bulls', 'city': 'Chicago', 'tricode': 'CHI', 'score': 110}, 
'winnerTricode': 'MIL', 'gameLeaders': {'home': {'name': 'Giannis Antetokounmpo', 'points': 41, 'rebounds': 15, 
'assists': 9}, 'away': {'name': 'Josh Giddey', 'points': 16, 'rebounds': 7, 'assists': 14}}}, {'gameId': 
'0022500036', 'gameStatus': 'Final', 'homeTeam': {'name': 'Timberwolves', 'city': 'Minnesota', 'tricode': 'MIN', 
'score': 137}, 'awayTeam': {'name': 'Jazz', 'city': 'Utah', 'tricode': 'UTA', 'score': 97}, 'winnerTricode': 'MIN',
'gameLeaders': {'home': {'name': 'Anthony Edwards', 'points': 37, 'rebounds': 5, 'assists': 1}, 'away': {'name': 
'Keyonte George', 'points': 18, 'rebounds': 4, 'assists': 2}}}, {'gameId': '0022500037', 'gameStatus': 'Final', 
'homeTeam': {'name': 'Nuggets', 'city': 'Denver', 'tricode': 'DEN', 'score': 129}, 'awayTeam': {'name': 'Warriors',
'city': 'Golden State', 'tricode': 'GSW', 

[Step 1: Duration 3.28 seconds| Input tokens: 2,730 | Output tokens: 24]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  formatted_results = format_summaries_to_text(game_summaries=game_summaries)                                      
  final_answer(formatted_results)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 🏀 Today's NBA Game Results 🏀

[Final] Magic (123) vs Celtics (110) (Winner: Magic)
  -> Home Leader: Franz Wagner (27 pts, 6 reb, 6 ast)
  -> Away Leader: Jaylen Brown (32 pts, 9 reb, 3 ast)
---
[Final] Wizards (114) vs Cavaliers (148) (Winner: Cavaliers)
  -> Home Leader: CJ McCollum (25 pts, 2 reb, 3 ast)
  -> Away Leader: Donovan Mitchell (24 pts, 5 reb, 5 ast)
---
[Final] Hawks (97) vs Raptors (109) (Winner: Raptors)
  -> Home Leader: Jalen Johnson (21 pts, 7 reb, 4 ast)
  -> Away Leader: Brandon Ingram (20 pts, 6 reb, 4 ast)
---
[Final] Nets (107) vs Pistons (125) (Winner: Pistons)
  -> Home Leader: Michael Porter Jr. (28 pts, 5 reb, 2 ast)
  -> Away Leader: Cade Cunningham (34 pts, 1 reb, 10 ast)
---
[Final] Spurs (121) vs Rockets (110) (Winner: Spurs)
  -> Home Leader: Victor Wembanyama (22 pts, 8 reb, 4 ast)
  -> Away Leader: Alperen Sengun (25 pts, 9 reb, 8 ast)
---
[Final] Heat (126) vs Hornets (108) (Winner: Heat)
  -> Home Leader: Jaime Jaquez Jr. (18 pts, 8 reb, 9 ast)
  -> Away Leader: Kon Knueppel (30 pts, 8 reb, 3 ast)
---
[Final] Grizzlies (118) vs Mavericks (104) (Winner: Grizzlies)
  -> Home Leader: Ja Morant (21 pts, 5 reb, 13 ast)
  -> Away Leader: Naji Marshall (16 pts, 7 reb, 4 ast)
---
[Final] Bucks (126) vs Bulls (110) (Winner: Bucks)
  -> Home Leader: Giannis Antetokounmpo (41 pts, 15 reb, 9 ast)
  -> Away Leader: Josh Giddey (16 pts, 7 reb, 14 ast)
---
[Final] Timberwolves (137) vs Jazz (97) (Winner: Timberwolves)
  -> Home Leader: Anthony Edwards (37 pts, 5 reb, 1 ast)
  -> Away Leader: Keyonte George (18 pts, 4 reb, 2 ast)
---
[Final] Nuggets (129) vs Warriors (104) (Winner: Nuggets)
  -> Home Leader: Nikola Jokić (26 pts, 9 reb, 9 ast)
  -> Away Leader: Draymond Green (17 pts, 6 reb, 4 ast)
---
[Final] Kings (101) vs Thunder (132) (Winner: Thunder)
  -> Home Leader: Russell Westbrook (24 pts, 6 reb, 9 ast)
  -> Away Leader: Isaiah Hartenstein (33 pts, 19 reb, 3 ast)
---

[Step 2: Duration 1.47 seconds| Input tokens: 7,399 | Output tokens: 58]

🏀 Today's NBA Game Results 🏀

[Final] Magic (123) vs Celtics (110) (Winner: Magic)
  -> Home Leader: Franz Wagner (27 pts, 6 reb, 6 ast)
  -> Away Leader: Jaylen Brown (32 pts, 9 reb, 3 ast)
---
[Final] Wizards (114) vs Cavaliers (148) (Winner: Cavaliers)
  -> Home Leader: CJ McCollum (25 pts, 2 reb, 3 ast)
  -> Away Leader: Donovan Mitchell (24 pts, 5 reb, 5 ast)
---
[Final] Hawks (97) vs Raptors (109) (Winner: Raptors)
  -> Home Leader: Jalen Johnson (21 pts, 7 reb, 4 ast)
  -> Away Leader: Brandon Ingram (20 pts, 6 reb, 4 ast)
---
[Final] Nets (107) vs Pistons (125) (Winner: Pistons)
  -> Home Leader: Michael Porter Jr. (28 pts, 5 reb, 2 ast)
  -> Away Leader: Cade Cunningham (34 pts, 1 reb, 10 ast)
---
[Final] Spurs (121) vs Rockets (110) (Winner: Spurs)
  -> Home Leader: Victor Wembanyama (22 pts, 8 reb, 4 ast)
  -> Away Leader: Alperen Sengun (25 pts, 9 reb, 8 ast)
---
[Final] Heat (126) vs Hornets (108) (Winner: Heat)
  -> Home Leader: Jaime Jaquez Jr. (18 pts, 8 reb, 9 ast)
  -

# Gradio

In [ ]:
def nba_chat(message: str, history: List[Dict[str, str]]):
    """
    This is the core function that Gradio's ChatInterface will call.
    'message' is the user's new input.
    'history' is now a list of OpenAI-style messages:
    [
        {"role": "user", "content": "Hello"},
        {"role": "assistant", "content": "Hi there!"}
    ]

    Our CodeAgent is stateless (it plans and executes based on the *current* query),
    so we only need to pass the 'message' to agent.run().
    """
    print(f"[User Query]: {message}")
    try:
        response = agent.run(message)
        print(f"[Agent Response]: {response}")
        return response
    except Exception as e:
        print(f"[Agent Error]: {e}")
        return f"An error occurred while processing your request: {e}"

iface = gr.ChatInterface(
    fn=nba_chat,
    title="🏀 NBA Chatbot 🏀",
    description="Ask me about today's NBA games, scores, and leaders. (Powered by smolagents & Gemini)",
    examples=[
        "What were today's NBA game results?",
        "Did the Celtics play today? Who was the game leader?",
        "LeBron James's career stats?"
    ],
    cache_examples=False,
    type="messages"
)

In [ ]:
print("Starting Gradio NBA Chatbot...")
iface.launch(share=False)

Starting Gradio NBA Chatbot...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>